# 1. XGBoost
**Use case:** Financial Services — Real-time credit card fraud detection

Every time a card is swiped or used online, the issuing bank has a few milliseconds to approve or decline. Only about 1-2% of authorizations are fraud, but each one costs the bank the full amount plus chargeback fees, while declining a good customer costs goodwill and call-centre time. The bank has rich history: amount, merchant category, channel, distance from home, velocity of recent transactions and account age. Success means catching more fraud dollars at the same or lower false-decline rate than today's rules.

> 💡 Think of a cricket team reviewing match footage: the first coach fixes the biggest batting flaw, the next coach looks only at what is still going wrong, and so on. After many small corrections the team is far better than any single coach could make it.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Score 2 new card swipes with 20 tiny trees
Train XGBoost on 18 hand-typed card authorizations and watch the fraud probability sharpen as trees are added.

| 5W + H | |
|---|---|
| **Who** | A fraud strategy analyst learning how the bank's authorization score is produced. |
| **What** | A tiny fraud model that says APPROVE or DECLINE/REVIEW for two new swipes. |
| **When** | During card authorization - the score must come back in milliseconds while the customer waits at the till. |
| **Where** | In the issuer's authorization path, between the card network request and the approve/decline response. |
| **Why** | To see that boosting is just many small trees adding up corrections, before scaling to millions of rows. |
| **How** | Type 18 swipes as Python lists, fit XGBClassifier with 20 depth-2 trees, and print the probability after 1, 3, 5, 10 and 20 trees. |

**🧠 ML concepts:** Boosting: each tree fixes the errors of earlier trees, Learning rate as a step size, Predicted probability vs hard decision, Feature importance, Decision threshold  
**🐍 Python concepts:** Lists of lists, List repetition and concatenation, numpy arrays, f-string alignment specs, zip and sorted with a lambda key, List comprehension with inline if/else

In [ ]:
# CONCEPT: XGBoost learns to spot card fraud from 18 hand-typed authorizations
import numpy as np
from xgboost import XGBClassifier

# 🐍 PY: a list of lists - each inner list is one swipe: [amount_usd, km_from_home, hour, is_online]
swipes = [
    [12.5, 2, 9, 0], [45.0, 5, 13, 0], [8.9, 1, 8, 0], [60.0, 3, 18, 0],
    [25.0, 4, 12, 1], [33.3, 6, 20, 1], [15.0, 2, 10, 0], [75.0, 8, 19, 0],
    [22.0, 3, 14, 1], [40.0, 5, 17, 0], [18.0, 1, 11, 0], [55.0, 7, 16, 1],
    [820.0, 3, 15, 0],
    [950.0, 4200, 3, 1], [780.0, 3100, 2, 1], [120.0, 5000, 4, 1], [640.0, 9, 1, 0],
    [35.0, 4800, 3, 1],
]
# 🐍 PY: [0] * 13 repeats a list; + joins lists -> 13 genuine swipes then 5 fraud swipes
labels = [0] * 13 + [1] * 5
# 🐍 PY: np.array turns Python lists into fast numeric arrays that ML libraries expect
X = np.array(swipes)
y = np.array(labels)
print("Data shape:", X.shape, "| fraud rows:", y.sum(), "| fraud rate:", f"{y.mean():.0%}")

# 🧠 ML: XGBoost builds many small trees; each new tree corrects the mistakes of all trees before it
# 🧠 ML: max_depth=2 keeps each tree tiny; learning_rate=0.3 means each tree only nudges the score
model = XGBClassifier(n_estimators=20, max_depth=2, learning_rate=0.3,
                      min_child_weight=0.5, random_state=42)
model.fit(X, y)

# Two brand-new swipes the model has never seen: a coffee and a 3am online purchase far from home
new_swipes = np.array([[30.0, 3, 12, 0], [880.0, 3900, 3, 1]])

# 🧠 ML: watch the fraud probability sharpen as more trees are added (iteration_range = use first k trees)
print("\nTrees used | P(fraud) coffee swipe | P(fraud) 3am online swipe")
for k in [1, 3, 5, 10, 20]:
    p = model.predict_proba(new_swipes, iteration_range=(0, k))[:, 1]
    # 🐍 PY: f-string format spec :>10 right-aligns in 10 characters, .3f = 3 decimals
    print(f"{k:>10} | {p[0]:>20.3f} | {p[1]:>25.3f}")

# 🧠 ML: feature importance = how much each column helped the trees separate fraud from genuine
# 🧠 ML: with only 18 rows one clean split can separate everything, so one feature may take all credit
names = ["amount_usd", "km_from_home", "hour_of_day", "is_online"]
print("\nFeature importance (gain share):")
# 🐍 PY: zip pairs names with scores; sorted(key=lambda ...) orders them, the minus sign = descending
for name, imp in sorted(zip(names, model.feature_importances_), key=lambda t: -t[1]):
    print(f"  {name:<13} {'#' * int(imp * 40):<40} {imp:.2f}")

# 🧠 ML: the bank chooses a cut-off; above it the authorization is declined or sent to step-up checks
probs = model.predict_proba(new_swipes)[:, 1]
# 🐍 PY: a list comprehension with an inline if/else builds one decision per swipe
decisions = ["DECLINE/REVIEW" if p > 0.5 else "APPROVE" for p in probs]
print("\nDecisions:", decisions)

**Try this next:**
- Change learning_rate to 1.0 and see how fast the probabilities jump - then try 0.05.
- Add a 6th fraud swipe that is small and local (e.g. [9.0, 2, 14, 0]) and see how the model reacts.
- Set max_depth=1 so every tree is a single question and compare the final probabilities.

## BUILD: Score 5,000 authorizations with 2% fraud
Train XGBoost on a realistic imbalanced card dataset and judge it with PR-AUC, thresholds and dollars stopped.

| 5W + H | |
|---|---|
| **Who** | A fraud data scientist building the first version of a card-not-present and card-present fraud score. |
| **What** | A fraud probability for every authorization plus a table showing caught fraud vs false alarms at three cut-offs. |
| **When** | Offline on last quarter's labelled authorizations, before the model is wired into real-time scoring. |
| **Where** | In a notebook or batch job reading from the fraud data mart; scores later move to the authorization API. |
| **Why** | Fraud is only ~2% of swipes, so accuracy is useless - the team must see recall, precision and dollars at stake. |
| **How** | Simulate 5,000 authorizations with a hidden non-linear fraud rule, stratified split, scale_pos_weight for imbalance, then PR-AUC and a threshold table. |

**🧠 ML concepts:** Class imbalance and scale_pos_weight, Stratified train/test split, ROC-AUC vs PR-AUC, Confusion matrix and precision, Threshold trade-off, Row and column subsampling  
**🐍 Python concepts:** numpy default_rng generator, pandas DataFrame from a dict, Boolean masks with & and ~, Tuple unpacking from ravel(), Conditional expression to avoid divide-by-zero, Series.items() loop

In [ ]:
# BUILD: XGBoost on 5,000 synthetic card authorizations with ~1-2% fraud
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier

# 🐍 PY: default_rng(42) is numpy's modern random generator; the seed makes results repeatable
rng = np.random.default_rng(42)
n = 5000

# 🐍 PY: a DataFrame built from a dict - each key becomes a column name
df = pd.DataFrame({
    "txn_amount": rng.lognormal(3.6, 1.0, n).round(2),
    "merchant_risk": rng.uniform(0, 1, n).round(3),
    "km_from_home": rng.exponential(25, n).round(1),
    "hour": rng.integers(0, 24, n),
    "card_present": rng.binomial(1, 0.7, n),
    "txns_last_24h": rng.poisson(3, n),
    "acct_age_days": rng.integers(30, 3650, n),
})

# Hidden "true" fraud pattern, including a non-linear interaction trees can discover
night = (df["hour"] < 5).astype(int)
far_cnp = ((df["km_from_home"] > 60) & (df["card_present"] == 0)).astype(int)
logit = (-10.0 + 0.8 * np.log1p(df["txn_amount"]) + 3.0 * df["merchant_risk"] + 1.8 * night
         + 3.5 * far_cnp + 0.3 * df["txns_last_24h"] - 0.0005 * df["acct_age_days"])
# 🧠 ML: the sigmoid turns a score into a probability; binomial draws 1 (fraud) with that probability
df["is_fraud"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"Rows: {len(df)}  fraud cases: {df.is_fraud.sum()}  fraud rate: {df.is_fraud.mean():.2%}")

X = df.drop(columns="is_fraud")
y = df["is_fraud"]
# 🧠 ML: stratify=y keeps the same tiny fraud % in train and test so the test set is not fraud-free
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# 🧠 ML: scale_pos_weight up-weights rare fraud rows (~ genuine count / fraud count) to fight imbalance
spw = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
                      n_jobs=2, random_state=42)
model.fit(X_train, y_train)
print(f"scale_pos_weight used: {spw:.1f}")

# 🐍 PY: [:, 1] slices column 1 = probability of the positive class (fraud)
proba = model.predict_proba(X_test)[:, 1]
# 🧠 ML: ROC-AUC ranks all cases; PR-AUC focuses on the rare class and is the honest metric for fraud
print(f"\nROC-AUC: {roc_auc_score(y_test, proba):.3f}")
print(f"PR-AUC : {average_precision_score(y_test, proba):.3f}  (random model = {y_test.mean():.3f})")

print("\nThreshold | caught fraud | missed | false alarms | precision")
for t in [0.5, 0.7, 0.9]:
    # 🐍 PY: .ravel() flattens the 2x2 matrix so we can unpack 4 numbers in one line
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= t).astype(int)).ravel()
    prec = tp / (tp + fp) if (tp + fp) else 0
    print(f"{t:>9} | {tp:>12} | {fn:>6} | {fp:>12} | {prec:>9.2f}")

# 🧠 ML: importance shows which signals drive the score - useful for analysts and model validators
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop drivers:")
for col, v in imp.head(5).items():
    print(f"  {col:<15} {'#' * int(v * 50)} {v:.2f}")

# Business view: dollars of fraud stopped at the 0.5 threshold
flag = proba >= 0.5
# 🐍 PY: boolean masks combine with & ; .sum() on dollars gives totals
caught = X_test.loc[flag & (y_test == 1).values, "txn_amount"].sum()
missed = X_test.loc[~flag & (y_test == 1).values, "txn_amount"].sum()
print(f"\nAt threshold 0.5: fraud $ stopped = ${caught:,.0f}, fraud $ missed = ${missed:,.0f}")
print(f"Customers inconvenienced (declines/step-up) = {int(flag.sum())} of {len(flag)} swipes")

**Try this next:**
- Remove scale_pos_weight and compare how many frauds are caught at 0.5.
- Add a feature 'is_night' = hour < 5 explicitly and see if PR-AUC changes.
- Try thresholds 0.2 and 0.3 - how many extra false alarms per extra fraud caught?

## PRODUCTION: Real-time fraud scorer with cost-based cut-off
Build a time-aware, tuned XGBoost pipeline that beats a logistic baseline, picks a threshold on dollars, and is saved for deployment.

| 5W + H | |
|---|---|
| **Who** | The fraud ML team plus model risk management, who must approve the model before it scores live traffic. |
| **What** | A saved pipeline (encoder + scaler + XGBoost) and a business-chosen threshold, with PR-AUC, permutation importance and governance notes. |
| **When** | Trained on the first 80% of a quarter and tested on the last 20%, mimicking deployment on future traffic. |
| **Where** | Batch training in the model platform; the joblib artifact is served behind a low-latency authorization API. |
| **Why** | Fraud patterns drift and false declines cost real money and goodwill, so tuning, time-aware validation and a cost-based cut-off are required. |
| **How** | ColumnTransformer + XGBoost Pipeline, RandomizedSearchCV over TimeSeriesSplit on PR-AUC, cost sweep, permutation importance, joblib.dump to a temp dir. |

**🧠 ML concepts:** Time-ordered validation to avoid leakage, Hyperparameter search on PR-AUC, Cost-sensitive threshold selection, Permutation importance, Baseline comparison, Model risk governance (SR 11-7)  
**🐍 Python concepts:** Functions with default arguments and docstrings, Pipeline step__param naming, min() with a key function, Dict comprehension to clean keys, if __name__ == '__main__' guard, tempfile and os.path

In [ ]:
# PRODUCTION: production-style XGBoost card-fraud scorer - time split, tuning, cost threshold, explainability
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
# 🐍 PY: UPPER_CASE names are a convention for module-level constants that never change
NUM = ["txn_amount", "km_from_home", "hour", "txns_last_1h", "amt_vs_avg", "acct_age_days", "card_present"]
CAT = ["merchant_cat", "channel"]
COST_FN, COST_FP = 1.0, 15.0  # missed fraud costs its amount; a false decline costs ~$15 friction + call


def make_data(n=20000, seed=42):
    """Simulate time-ordered card authorizations with ~1.5% fraud."""
    rng = np.random.default_rng(seed)
    # 🐍 PY: a dict comprehension is not needed here - a plain dict literal maps column -> array
    df = pd.DataFrame({
        "ts": np.sort(rng.uniform(0, 90, n)),  # day number over one quarter
        "txn_amount": rng.lognormal(3.7, 1.1, n).round(2),
        "km_from_home": rng.exponential(30, n).round(1),
        "hour": rng.integers(0, 24, n),
        "txns_last_1h": rng.poisson(0.6, n),
        "amt_vs_avg": rng.lognormal(0, 0.6, n).round(2),
        "acct_age_days": rng.integers(15, 4000, n),
        "card_present": rng.binomial(1, 0.65, n),
        "merchant_cat": rng.choice(["grocery", "fuel", "electronics", "travel", "gift_cards", "dining"],
                                   n, p=[.30, .15, .12, .10, .05, .28]),
        "channel": rng.choice(["pos", "ecom", "atm"], n, p=[.6, .3, .1]),
    })
    # 🐍 PY: .isin() returns True/False per row; .astype(int) turns that into 1/0 flags
    risky_mcc = df["merchant_cat"].isin(["electronics", "gift_cards"]).astype(int)
    burst = (df["txns_last_1h"] >= 3).astype(int)
    far_cnp = ((df["km_from_home"] > 80) & (df["card_present"] == 0)).astype(int)
    # Fraud signals combine non-linearly (rings hit risky merchants at night) - trees capture this
    combo = risky_mcc * (df["hour"] < 6) * (df["channel"] == "ecom")
    logit = (-6.4 + 1.2 * np.log(df["amt_vs_avg"] + 0.1) + 0.8 * risky_mcc + 2.2 * burst
             + 3.0 * far_cnp + 3.5 * combo + 0.6 * (df["hour"] < 5) + 0.3 * (df["channel"] == "ecom")
             - 0.0004 * df["acct_age_days"] + 0.35 * np.log1p(df["txn_amount"]))
    df["is_fraud"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return df


def build_pipeline(model):
    # 🧠 ML: one Pipeline = identical preprocessing in training and in the real-time scoring API
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])
    return Pipeline([("pre", pre), ("clf", model)])


def best_threshold(y, proba, amounts):
    """Pick the cut-off that minimises total business cost on validation data."""
    rows = []
    for t in np.arange(0.05, 0.96, 0.05):
        flag = proba >= t
        # 🧠 ML: cost-sensitive thresholding - a missed $900 fraud hurts more than a missed $5 one
        fn_cost = amounts[(~flag) & (y == 1)].sum() * COST_FN
        fp_cost = ((flag) & (y == 0)).sum() * COST_FP
        rows.append((round(t, 2), fn_cost + fp_cost, int((flag & (y == 1)).sum()), int((flag & (y == 0)).sum())))
    # 🐍 PY: min with key=lambda picks the tuple whose 2nd item (total cost) is smallest
    return min(rows, key=lambda r: r[1]), rows


def main():
    df = make_data()
    print(f"Authorizations: {len(df):,}  fraud rate: {df.is_fraud.mean():.2%}")
    # 🧠 ML: time-ordered split - train on the past, test on the future (no shuffling = no leakage)
    # 🐍 PY: .iloc slices by row position - first 80% for training, last 20% for testing
    cut = int(len(df) * 0.8)
    train, test = df.iloc[:cut], df.iloc[cut:]
    X_tr, y_tr, X_te, y_te = train[NUM + CAT], train.is_fraud, test[NUM + CAT], test.is_fraud

    # Baseline: logistic regression with class weighting
    base = build_pipeline(LogisticRegression(max_iter=500, class_weight="balanced"))
    base.fit(X_tr, y_tr)
    # 🧠 ML: always keep a simple baseline - the complex model must beat it to justify its cost
    p_base = base.predict_proba(X_te)[:, 1]

    # 🧠 ML: scale_pos_weight handles the ~1:65 imbalance instead of throwing away genuine rows
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    xgb = build_pipeline(XGBClassifier(n_estimators=150, tree_method="hist", eval_metric="aucpr",
                                       scale_pos_weight=spw, n_jobs=2, random_state=42))
    # 🐍 PY: keys use the 'step__param' naming so the search reaches inside the Pipeline
    grid = {"clf__max_depth": [3, 4, 5], "clf__learning_rate": [0.03, 0.06, 0.1],
            "clf__subsample": [0.7, 0.9], "clf__min_child_weight": [1, 5]}
    # 🧠 ML: TimeSeriesSplit cross-validation respects time order; PR-AUC is the score for rare fraud
    search = RandomizedSearchCV(xgb, grid, n_iter=4, cv=TimeSeriesSplit(n_splits=3),
                                scoring="average_precision", random_state=42, n_jobs=1)
    search.fit(X_tr, y_tr)
    print("Best params:", {k.replace("clf__", ""): v for k, v in search.best_params_.items()})
    model = search.best_estimator_
    p_xgb = model.predict_proba(X_te)[:, 1]

    # 🐍 PY: looping over a list of (name, value) tuples keeps the comparison table compact
    print("\nModel               ROC-AUC   PR-AUC")
    for name, p in [("LogReg baseline", p_base), ("XGBoost tuned", p_xgb)]:
        print(f"{name:<18} {roc_auc_score(y_te, p):>8.3f} {average_precision_score(y_te, p):>8.3f}")
    print(f"(PR-AUC of random guessing = {y_te.mean():.3f})")

    # Threshold from business cost (in real life tune on a separate validation month)
    amounts = X_te["txn_amount"].values
    (t, cost, tp, fp), rows = best_threshold(y_te.values, p_xgb, amounts)
    print("\nThreshold | total cost $ | fraud caught | false declines")
    for r in rows[::3]:
        print(f"{r[0]:>9} | {r[1]:>12,.0f} | {r[2]:>12} | {r[3]:>14}")
    no_model = amounts[y_te.values == 1].sum()
    print(f"Chosen threshold {t}: cost ${cost:,.0f} vs ${no_model:,.0f} with no model")

    # 🧠 ML: permutation importance = drop in PR-AUC when one raw column is shuffled (model-agnostic)
    pi = permutation_importance(model, X_te, y_te, scoring="average_precision",
                                n_repeats=3, random_state=42, n_jobs=1)
    # 🐍 PY: argsort of the negative gives indices from largest to smallest
    order = np.argsort(-pi.importances_mean)
    print("\nPermutation importance (PR-AUC drop):")
    for i in order[:6]:
        print(f"  {X_te.columns[i]:<14} {pi.importances_mean[i]:+.4f}")

    # 🧠 ML: model-risk note - document, validate independently, monitor drift (SR 11-7 / OCC 2011-12)
    print("\nModel risk: SR 11-7 validation, monthly PSI drift checks, reason codes for declines,")
    print("PCI-DSS for card data; fraud models are not credit decisions but still need governance.")

    out = tempfile.mkdtemp()
    path = os.path.join(out, "card_fraud_xgb.joblib")
    # 🐍 PY: joblib.dump pickles the whole pipeline (scaler + encoder + model) into one file
    joblib.dump({"pipeline": model, "threshold": t}, path)
    print(f"Saved model + threshold to {os.path.basename(path)} ({os.path.getsize(path) // 1024} KB)")


# 🐍 PY: this guard runs main() only when the file is executed, not when it is imported
if __name__ == "__main__":
    main()

**Try this next:**
- Change COST_FP to 5 (a cheap step-up SMS instead of a decline) and see how the chosen threshold moves.
- Set n_iter=8 in RandomizedSearchCV and check whether PR-AUC improves enough to justify the extra time.
- Shuffle the split (train_test_split) instead of the time split and compare - optimistic results are a sign of leakage.